# Import Libraries

In [36]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
import re
import tensorflow as tf
from models.llama3.generation import Llama
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
import requests
from openai import OpenAI

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [2]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

In [3]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset3.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)

# Question Classifier

## Text-Classification

In [8]:
tokenizer = AutoTokenizer.from_pretrained("uw-vta/bloominzer-0.1")
model = AutoModelForSequenceClassification.from_pretrained("uw-vta/bloominzer-0.1")
bloominzer = pipeline("text-classification", model=model, tokenizer=tokenizer)

Device set to use mps:0


In [15]:
message = bloominzer("How many total disk access is needed to search a record using two level indexing?")
print(message[0]['label'])

Knowledge


### Assign Prediction Labels

In [9]:
pred_labels= []
for query in tqdm(queries):
    message = bloominzer(query)
    pred_labels.append(message[0]['label'])

100%|██████████| 181/181 [00:04<00:00, 41.42it/s]


In [10]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        23
           1       0.25      0.86      0.39        37
           2       0.50      0.07      0.12        29
           3       1.00      0.07      0.12        30
           4       0.50      0.03      0.06        29
           5       0.34      0.48      0.40        33

    accuracy                           0.29       181
   macro avg       0.43      0.25      0.18       181
weighted avg       0.44      0.29      0.20       181



## LLM Classification

## Zero-Shot

### BART

In [11]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli")

bart = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [13]:
sequence_to_classify = "How many total disk access is needed to search a record using two level indexing?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
p_label = bart(sequence_to_classify, candidate_labels)
p_label['labels'][0]

'analysis'

### Assign Prediction Label

In [12]:
pred_labels= []
for query in tqdm(queries):
    sequence_to_classify = query
    candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
    p_label = bart(sequence_to_classify, candidate_labels)
    pred_labels.append(p_label['labels'][0])

100%|██████████| 181/181 [00:24<00:00,  7.39it/s]


In [13]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.23      0.13      0.17        23
           1       0.44      0.22      0.29        37
           2       0.52      0.79      0.63        29
           3       0.33      0.77      0.46        30
           4       0.53      0.34      0.42        29
           5       0.65      0.33      0.44        33

    accuracy                           0.43       181
   macro avg       0.45      0.43      0.40       181
weighted avg       0.46      0.43      0.40       181



### mDeBERTa-v3-base-mnli-xnli

In [14]:
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
model = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ya_classifier = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [11]:
sequence_to_classify = "How many total disk access is needed to search a record using two level indexing?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
label = ya_classifier(sequence_to_classify, candidate_labels)
label['labels'][0]

'evaluation'

### Assign Prediction Label

In [15]:
pred_labels= []
for query in tqdm(queries):
    sequence_to_classify = query
    candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
    p_label = ya_classifier(sequence_to_classify, candidate_labels)
    pred_labels.append(p_label['labels'][0])


100%|██████████| 181/181 [00:28<00:00,  6.45it/s]


In [16]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.23      0.48      0.31        23
           1       0.35      0.16      0.22        37
           2       0.33      0.52      0.41        29
           3       0.42      0.60      0.49        30
           4       0.40      0.07      0.12        29
           5       0.70      0.48      0.57        33

    accuracy                           0.38       181
   macro avg       0.40      0.39      0.35       181
weighted avg       0.41      0.38      0.35       181



## Text Generation

### OWEN

In [16]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

qwen_classifier = pipeline("text-generation", model=model , tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.38s/it]
Device set to use mps:0


In [17]:
query = 'How many total disk access is needed to search a record using two level indexing?'

messages = [
    {
        "role": "user", 
            "content": f'''
            Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            Consider these definitions:
            1. **knowledge**: Recalling facts/definitions (who, what, when, where)
            2. **comprehension**: Explaining concepts in own words (summarize, describe)
            3. **application**: Using knowledge in new situations (solve, compute, demonstrate)
            4. **analysis**: Breaking down concepts (compare, contrast, categorize)
            5. **synthesis**: Creating new patterns/solutions (design, develop, integrate)
            6. **evaluation**: Making judgments with criteria (justify, critique, recommend)

            Query: "{query}"

            Decision rules:
            - Focus on the query's PRIMARY cognitive demand
            - For multiple operations, choose the HIGHEST applicable level

            Respond ONLY with the exact taxonomy word in lowercase. No punctuation.
            '''
    }
]
message = qwen_classifier(messages)

print(message[0]['generated_text'][1]['content'])

application


### Predict Label Assignment

In [18]:
# Reinforce 1
def q_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            REVISE YOUR CLASSIFICATION. Your previous response '{prev_res}' was INVALID. 
            Classify this query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            CRITICAL RULES:
            1. MUST select from the 6 specified terms - NO exceptions
            2. Use these precise definitions:
            - knowledge: Recalling facts, terms, basic concepts (identify, list, name)
            - comprehension: Explaining meaning (describe, discuss, summarize)
            - application: Using information in new situations (execute, implement, solve)
            - analysis: Drawing connections among ideas (differentiate, organize, attribute)
            - synthesis: Producing new patterns (design, construct, integrate)
            - evaluation: Making judgments with evidence (appraise, defend, recommend)
            3. If multiple levels apply, choose the HIGHEST appropriate level
            4. Respond ONLY with the lowercase taxonomy word - NO other text

            Query: "{query}"

            Re-evaluate carefully. Your response MUST be exactly one word from the list.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [19]:
# Reinforce 2
def q1_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            Predict Bloom's Taxonomy level of understanding. Classify query: {query} in one word. Responding {prev_res} is danger.
            Your entire response must be just one word chosen from following: 
            {label_mapper.keys()}
            {prev_res} is incorrect.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [20]:
pred_labels= []
for query in tqdm(queries):
    messages = [
        {
            "role": "user", 
            "content": f'''
            Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            Consider these definitions:
            1. **knowledge**: Recalling facts/definitions (who, what, when, where)
            2. **comprehension**: Explaining concepts in own words (summarize, describe)
            3. **application**: Using knowledge in new situations (solve, compute, demonstrate)
            4. **analysis**: Breaking down concepts (compare, contrast, categorize)
            5. **synthesis**: Creating new patterns/solutions (design, develop, integrate)
            6. **evaluation**: Making judgments with criteria (justify, critique, recommend)

            Query: "{query}"

            Decision rules:
            - Focus on the query's PRIMARY cognitive demand
            - For multiple operations, choose the HIGHEST applicable level

            Respond ONLY with the exact taxonomy word in lowercase. No punctuation.
            ''',
        }
    ]

    message = qwen_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'][1]['content'])

while(len(set(pred_labels)) != 6):
    for i , query in enumerate(queries):
        if(pred_labels[i].lower() not in label_mapper.keys()):
            prev_res = pred_labels[i]
            print(prev_res)
            pred_labels[i] = q_classifier(query, prev_res)
            if(pred_labels[i].lower() not in label_mapper.keys()):
                pred_labels[i] = q1_classifier(query, prev_res)
    print(set(pred_labels))

100%|██████████| 181/181 [02:24<00:00,  1.26it/s]


definition
definition
definition
definition
definition
evaluation
definition
definition
definition
{'definition', 'application', 'synthesis', 'knowledge', 'definition is incorrect.', 'comprehension', 'evaluation', 'analysis'}
definition
definition
definition is incorrect.
{'application', 'synthesis', 'knowledge', 'comprehension', 'evaluation', 'analysis'}


In [22]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.30      0.47        23
           1       0.71      0.14      0.23        37
           2       0.32      0.41      0.36        29
           3       0.34      0.77      0.47        30
           4       0.65      0.59      0.62        29
           5       0.61      0.67      0.64        33

    accuracy                           0.48       181
   macro avg       0.61      0.48      0.46       181
weighted avg       0.60      0.48      0.46       181



## Google-FLAN-T5-XL

In [23]:
model_name = "google/flan-t5-xl"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

flan_classifier = pipeline("text2text-generation", model=model , tokenizer=tokenizer, device=-1)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 13.48it/s]
Device set to use cpu


In [24]:
query = 'How many total disk access is needed to search a record using two level indexing?'

messages = f"""
    Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

    **Bloom's Taxonomy Levels:**
    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
    - Keywords: define, list, memorize, recall, repeat
    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
    - Keywords: describe, explain, paraphrase, summarize
    3. Application: Using learned information in new concrete situations to solve problems
    - Keywords: apply, demonstrate, solve, use, implement
    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
    - Keywords: analyze, compare, contrast, differentiate, examine
    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
    - Keywords: create, design, propose, formulate, integrate
    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
    - Keywords: assess, critique, defend, evaluate, justify

    **Classification Rules:**
    - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
    - Choose the HIGHEST level that substantially applies
    - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
    - If multiple levels apply, select the most complex

    **Query to Classify:**
    "{query}"
        """

message = flan_classifier(messages)
print(message)

[{'generated_text': 'application'}]


### Predict Label Assignment

In [25]:
pred_labels= []
for query in tqdm(queries):
    messages = f"""
        Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

        **Bloom's Taxonomy Levels:**
        1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
        - Keywords: define, list, memorize, recall, repeat
        2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
        - Keywords: describe, explain, paraphrase, summarize
        3. Application: Using learned information in new concrete situations to solve problems
        - Keywords: apply, demonstrate, solve, use, implement
        4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
        - Keywords: analyze, compare, contrast, differentiate, examine
        5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
        - Keywords: create, design, propose, formulate, integrate
        6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
        - Keywords: assess, critique, defend, evaluate, justify

        **Classification Rules:**
        - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
        - Choose the HIGHEST level that substantially applies
        - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
        - If multiple levels apply, select the most complex

        **Query to Classify:**
        "{query}"
    """

    message = flan_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'].lower())

100%|██████████| 181/181 [1:35:40<00:00, 31.71s/it]


In [26]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       0.79      0.48      0.59        23
           1       1.00      0.14      0.24        37
           2       0.44      0.93      0.59        29
           3       0.78      0.60      0.68        30
           4       0.41      0.90      0.56        29
           5       0.85      0.33      0.48        33

    accuracy                           0.54       181
   macro avg       0.71      0.56      0.52       181
weighted avg       0.72      0.54      0.51       181

